# 06 - Next Trading Day Signal and Reporting Engine

Goal: generate next-day buy/sell/hold indications, inspect history with candlestick charts, and export an investment impact report.

The dataset is daily historical market data, so "24 hrs" means the next available trading day after the latest row in the dataset.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import joblib
import pandas as pd
import plotly.express as px

from invest_intel.data import build_returns_matrix, load_prices, load_processed
from invest_intel.features import drop_modeling_na, modeling_feature_columns
from invest_intel.modeling import train_direction_model, train_return_models
from invest_intel.portfolio import construct_profile_portfolio
from invest_intel.reporting import build_investment_report, save_investment_report
from invest_intel.risk import risk_summary
from invest_intel.signals import signals_from_saved_model
from invest_intel.visualization import candlestick_with_volume

## Train One-Day Return and Direction Models

In [3]:
features = load_processed('nifty50_features.csv')
feature_cols = modeling_feature_columns(features)

return_df = drop_modeling_na(features, 'future_return_1d', feature_cols)
return_results = train_return_models(return_df, feature_cols, target_col='future_return_1d', test_start_date='2019-01-01')
metrics = pd.DataFrame([{'model': result.name, **result.metrics} for result in return_results])
metrics

,model,mae,rmse,r2,directional_accuracy
0,ridge_regression,0.016842,0.026287,-0.011339,0.500071
1,hist_gradient_boosting,0.016923,0.026376,-0.018232,0.507157


In [4]:
best = max(return_results, key=lambda result: result.metrics['directional_accuracy'])
(ROOT / 'models').mkdir(exist_ok=True)
joblib.dump({'model': best.estimator, 'feature_cols': feature_cols, 'target': 'future_return_1d'}, ROOT / 'models' / 'stock_return_1d_model.joblib')

direction_df = drop_modeling_na(features, 'target_direction_1d', feature_cols)
direction_result = train_direction_model(direction_df, feature_cols, target_col='target_direction_1d')
joblib.dump({'model': direction_result.estimator, 'feature_cols': feature_cols, 'target': 'target_direction_1d'}, ROOT / 'models' / 'stock_direction_1d_model.joblib')
direction_result.metrics

{'directional_accuracy': 0.5071570294784581}

## Generate Buy/Sell/Hold Signals

In [5]:
signals = signals_from_saved_model(features)
signals.head(25)

,date,symbol,close,predicted_return_1d,predicted_price_1d,technical_score,signal,confidence,signal_reason,signal_source
39,2021-04-30,SBIN,353.50,-0.020220,346.352387,0.4,STRONG SELL,0.865552,close above 20-day MA; close below 50-day MA; ...,model_plus_indicators
46,2021-04-30,TCS,3035.65,-0.005953,3017.578471,-2.8,SELL,0.819956,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
4,2021-04-30,BAJAJFINSV,11041.65,-0.012606,10902.462688,2.3,SELL,0.736102,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
44,2021-04-30,TATAMOTORS,293.85,-0.000906,293.583744,-2.8,SELL,0.729110,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
24,2011-06-28,INFOSYSTCH,2865.30,0.000640,2867.133169,2.8,BUY,0.724316,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
11,2021-04-30,DRREDDY,5163.10,0.000571,5166.048987,2.8,BUY,0.723081,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
31,2021-04-30,M&M,752.55,-0.000569,752.121457,-2.8,SELL,0.723050,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
32,2021-04-30,MARUTI,6455.65,-0.000242,6454.090656,-2.8,SELL,0.717148,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
50,2013-10-22,UNIPHOS,157.80,0.000118,157.818572,2.8,BUY,0.714918,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
27,2021-04-30,ITC,202.60,-0.000102,202.579307,-2.8,SELL,0.714638,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators


In [6]:
px.bar(
    signals.head(25),
    x='symbol',
    y='predicted_return_1d',
    color='signal',
    hover_data=['confidence', 'signal_reason'],
    title='Next trading day signal leaderboard',
)

## Candlestick History

In [7]:
prices = load_prices()
symbol = 'RELIANCE'
recent_prices = prices[(prices['symbol'] == symbol) & (prices['date'] >= '2020-01-01')]
candlestick_with_volume(recent_prices, symbol, title=f'{symbol} recent candlestick history')

## Portfolio Impact Report

In [8]:
returns = build_returns_matrix(prices[prices['date'] >= '2015-01-01'], min_obs=252).dropna(how='all')
risk = risk_summary(returns)
weights, metrics, selected = construct_profile_portfolio(returns, profile='balanced')
report = build_investment_report(signals, risk, selected)
path = save_investment_report(report)
display(report.head(25))
path

,date,symbol,signal,confidence,predicted_return_1d,predicted_price_1d,close,weight,portfolio_impact_score,annualized_return,annualized_volatility,sharpe_ratio,sortino_ratio,max_drawdown,var_95,expected_shortfall_95,signal_reason,signal_source
2,2021-04-30,BAJAJFINSV,SELL,0.736102,-0.012606,10902.462688,11041.65,0.148977,-0.001382,0.413608,0.364003,0.971442,1.326201,-0.585896,-0.031063,-0.049115,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
35,2021-04-30,TITAN,HOLD,0.616711,-0.000573,1490.795501,1491.65,0.099839,-0.000035,0.246423,0.328475,0.567540,0.851798,-0.417300,-0.028412,-0.043129,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
34,2021-04-30,ASIANPAINT,HOLD,0.616781,-0.000577,2534.937179,2536.40,0.099110,-0.000035,0.216985,0.265990,0.590192,0.880887,-0.288550,-0.023899,-0.035366,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
38,2021-04-30,HINDUNILVR,HOLD,0.604160,0.000124,2354.042952,2353.75,0.097076,0.000007,0.200041,0.237602,0.589394,0.995604,-0.213217,-0.020818,-0.029141,close below 20-day MA; close above 50-day MA; ...,model_plus_indicators
24,2021-04-30,BAJFINANCE,BUY,0.695677,0.000315,5453.619459,5451.90,0.089725,0.000020,0.073988,0.540944,0.025859,0.023358,-0.932820,-0.036420,-0.069521,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
17,2021-04-30,SHREECEM,SELL,0.703898,0.000495,27924.302930,27910.50,0.080609,0.000028,0.191990,0.312928,0.421792,0.673155,-0.368969,-0.028938,-0.041025,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
19,2021-04-30,NESTLEIND,SELL,0.703459,0.000519,16317.713485,16309.25,0.075210,0.000027,0.164980,0.255114,0.411501,0.654740,-0.324011,-0.021323,-0.032935,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
32,2021-04-30,TATASTEEL,HOLD,0.631458,0.000548,1034.566288,1034.00,0.065741,0.000023,0.163166,0.390693,0.264059,0.394455,-0.676133,-0.037441,-0.054760,close above 20-day MA; close above 50-day MA; ...,model_plus_indicators
14,2021-04-30,ULTRACEMCO,SELL,0.708855,0.000219,6280.326257,6278.95,0.063437,0.000010,0.148144,0.292208,0.301650,0.449343,-0.376561,-0.027575,-0.038734,close below 20-day MA; close below 50-day MA; ...,model_plus_indicators
40,2021-04-30,RELIANCE,SELL,0.601346,-0.005541,1983.447635,1994.50,0.062210,-0.000207,0.139186,0.363351,0.217932,0.226786,-0.526756,-0.025607,-0.045384,close above 20-day MA; close below 50-day MA; ...,model_plus_indicators


WindowsPath('E:/Projects/cult_invest_intel_ml/reports/investment_signal_report.csv')